In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    roc_curve,
    roc_auc_score)

## Carga de datos y eliminación de la fuga de información

In [ ]:
df = pd.read_csv("data/raw/dataset_practica_final.csv")

y_heart = df["is_canceled"]
X_heart = df.drop(columns=["is_canceled", "reservation_status", "reservation_status_date"]) #Elimino reservation_status y reservation_status_date porque se registran después de que la reserva se cancela o se completa. en el momento de predecir no estarían disponibles y el modelo aprendería a leer la respuesta

In [ ]:
meses = {"January":1, "February":2, "March":3, "April":4, "May":5, "June":6,
         "July":7, "August":8, "September":9, "October":10, "November":11, "December":12}

df["arrival_month_num"] = df["arrival_date_month"].map(meses)
df["arrival_date"] = pd.to_datetime(dict(year=df["arrival_date_year"],month=df["arrival_month_num"],day=df["arrival_date_day_of_month"])) # Unifico dia,mes y año y convierto a entero los meses del año

## Pipeline de preprocesado

In [ ]:
cat_cols = X_heart.select_dtypes(include=["object", "str"]).columns.tolist()
num_cols = [c for c in X_heart.columns if c not in cat_cols]

# Pipeline:
preproceso = ColumnTransformer([
    ("cat", Pipeline([
        ("imputar", SimpleImputer(strategy="constant", fill_value="Desconocido")),
        ("codificar", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), cat_cols),
    ("num", SimpleImputer(strategy="median"), num_cols),
])

## División en entrenamiento y test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_heart, y_heart, test_size=0.2, stratify=y_heart, random_state=42)

---
# Entrenamientos previos hasta la elección final

Antes de fijar los hiperparámetros definitivos probé varias configuraciones. Las
descartadas quedan aquí documentadas porque explican **por qué** elegí las que
finalmente usé.

Cada bloque imprime también la accuracy sobre **train**: comparada con la de
test, es la que revela si el modelo sobreajusta o subajusta.

## Random Forest v1 — árboles cortos



In [ ]:
modelo_rf_v1 = Pipeline([
    ("preproceso", clone(preproceso)),
    ("clasificador", RandomForestClassifier(n_estimators=150, max_depth=3,
                                            n_jobs=-1, random_state=42)),
])

modelo_rf_v1.fit(X_train,y_train)
print(modelo_rf_v1.score(X_test,y_test))

y_pred = modelo_rf_v1.predict(X_test)

# Obtención de las métricas de evaluación
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy en train: {modelo_rf_v1.score(X_train,y_train):.2%}")
print(f"Accuracy:  {acc:.2%}")
print(f"Precisión: {prec:.2%}")
print(f"Recall:    {rec:.2%}")
print(f"F1-Score:  {f1:.2%}")
print(f"ROC-AUC: {roc_auc_score(y_test, modelo_rf_v1.predict_proba(X_test)[:,1]):.4f}")

**Descartada.** Con `max_depth=3` cada árbol solo puede hacer tres preguntas
antes de decidir, así que no puede combinar `lead_time` con `deposit_type` con
`country`. El resultado lo deja claro:

- Precisión 99,54 % — cuando dice "cancela", casi nunca se equivoca.
- Recall 36,74 % — pero detecta solo un tercio de las cancelaciones.

Se ha vuelto extremadamente conservador: solo marca los casos clarísimos.

La accuracy en train (76,61 %) y en test (76,51 %) son casi idénticas, así que
no hay sobreajuste: hay **subajuste**. El modelo es demasiado simple para el
problema.

## Random Forest v2 — subiendo la profundidad

In [ ]:
modelo_rf_v2 = Pipeline([
    ("preproceso", clone(preproceso)),
    ("clasificador", RandomForestClassifier(n_estimators=300, max_depth=10,
                                            n_jobs=-1, random_state=42)),
])

modelo_rf_v2.fit(X_train,y_train)
print(modelo_rf_v2.score(X_test,y_test))

y_pred = modelo_rf_v2.predict(X_test)

# Obtención de las métricas de evaluación
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy en train: {modelo_rf_v2.score(X_train,y_train):.2%}")
print(f"Accuracy:  {acc:.2%}")
print(f"Precisión: {prec:.2%}")
print(f"Recall:    {rec:.2%}")
print(f"F1-Score:  {f1:.2%}")
print(f"ROC-AUC: {roc_auc_score(y_test, modelo_rf_v2.predict_proba(X_test)[:,1]):.4f}")

**Mejora clara pero insuficiente**: el F1 pasa del 53,67 % al 77,21 % y el
recall del 37 % al 68 %. Sigue habiendo subajuste (train 85,54 % frente a test
85,07 %).

Conclusión: el problema no era el número de árboles, era la **profundidad**. En
un Random Forest el sobreajuste lo contiene el promedio de árboles
decorrelacionados, no la poda de cada uno, así que limitar `max_depth` es
contraproducente.

## Random Forest v3 — profundidad libre

Abandono `max_depth` y regulo con `min_samples_leaf=10`.

In [ ]:
modelo_rf_v3 = Pipeline([
    ("preproceso", clone(preproceso)),
    ("clasificador", RandomForestClassifier(n_estimators=300, min_samples_leaf=10,
                                            max_depth=None, n_jobs=-1, random_state=42)),
])

modelo_rf_v3.fit(X_train,y_train)
print(modelo_rf_v3.score(X_test,y_test))

y_pred = modelo_rf_v3.predict(X_test)

# Obtención de las métricas de evaluación
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy en train: {modelo_rf_v3.score(X_train,y_train):.2%}")
print(f"Accuracy:  {acc:.2%}")
print(f"Precisión: {prec:.2%}")
print(f"Recall:    {rec:.2%}")
print(f"F1-Score:  {f1:.2%}")
print(f"ROC-AUC: {roc_auc_score(y_test, modelo_rf_v3.predict_proba(X_test)[:,1]):.4f}")

El F1 sube al 82,31 % y el recall al 76,81 %. `min_samples_leaf` exige que
cada hoja tenga al menos 10 reservas, lo que impide crear reglas para casos
aislados.

## Random Forest v4 — configuración final

In [ ]:
modelo_rf_v4 = Pipeline([
    ("preproceso", clone(preproceso)),
    ("clasificador", RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                            n_jobs=-1, random_state=42)),
])

modelo_rf_v4.fit(X_train,y_train)
print(modelo_rf_v4.score(X_test,y_test))

y_pred = modelo_rf_v4.predict(X_test)

# Obtención de las métricas de evaluación
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy en train: {modelo_rf_v4.score(X_train,y_train):.2%}")
print(f"Accuracy:  {acc:.2%}")
print(f"Precisión: {prec:.2%}")
print(f"Recall:    {rec:.2%}")
print(f"F1-Score:  {f1:.2%}")
print(f"ROC-AUC: {roc_auc_score(y_test, modelo_rf_v4.predict_proba(X_test)[:,1]):.4f}")

**Configuración elegida**: F1 84,86 %, recall 81,05 %, ROC-AUC 0,9595.

Obsérvese la accuracy en train (97,34 %) frente a la de test (89,29 %): ocho
puntos de distancia. Eso indica cierto sobreajuste, y es normal en un Random
Forest. Cada árbol memoriza algo del train, pero al promediar 300 árboles que
memorizan cosas distintas los errores se cancelan entre sí.

## XGBoost v1 — primera configuracion

Primer intento con el learning rate por defecto de XGBoost (0,3) y árboles
cortos.

In [ ]:
modelo_xgb_v1 = Pipeline([
    ("preproceso", clone(preproceso)),
    ("clasificador", XGBClassifier(n_estimators=100, learning_rate=0.3,
                                   max_depth=3, n_jobs=-1, random_state=42)),
])

modelo_xgb_v1.fit(X_train,y_train)
print(modelo_xgb_v1.score(X_test,y_test))

y_pred = modelo_xgb_v1.predict(X_test)

# Obtención de las métricas de evaluación
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy en train: {modelo_xgb_v1.score(X_train,y_train):.2%}")
print(f"Accuracy:  {acc:.2%}")
print(f"Precisión: {prec:.2%}")
print(f"Recall:    {rec:.2%}")
print(f"F1-Score:  {f1:.2%}")
print(f"ROC-AUC: {roc_auc_score(y_test, modelo_xgb_v1.predict_proba(X_test)[:,1]):.4f}")

F1 80,64 %. Un learning rate alto avanza a pasos grandes y se pasa de
largo; con solo 100 árboles no hay margen para corregir.

## XGBoost v2 — más árboles, learning rate menor

In [ ]:
modelo_xgb_v2 = Pipeline([
    ("preproceso", clone(preproceso)),
    ("clasificador", XGBClassifier(n_estimators=300, learning_rate=0.1,
                                   max_depth=6, n_jobs=-1, random_state=42)),
])

modelo_xgb_v2.fit(X_train,y_train)
print(modelo_xgb_v2.score(X_test,y_test))

y_pred = modelo_xgb_v2.predict(X_test)

# Obtención de las métricas de evaluación
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy en train: {modelo_xgb_v2.score(X_train,y_train):.2%}")
print(f"Accuracy:  {acc:.2%}")
print(f"Precisión: {prec:.2%}")
print(f"Recall:    {rec:.2%}")
print(f"F1-Score:  {f1:.2%}")
print(f"ROC-AUC: {roc_auc_score(y_test, modelo_xgb_v2.predict_proba(X_test)[:,1]):.4f}")

**Mejor resultado de XGBoost**: F1 83,68 % y ROC-AUC 0,9543.

## Resumen del proceso de ajuste

| Configuración | Hiperparámetros | Train acc | Test acc | F1 | ROC-AUC |
| --- | --- | --- | --- | --- | --- |
| RF v1 | `max_depth=3` | 76,61 % | 76,51 % | 53,67 % | 0,8708 |
| RF v2 | `max_depth=10` | 85,54 % | 85,07 % | 77,21 % | 0,9311 |
| RF v3 | `min_samples_leaf=10` | 89,95 % | 87,77 % | 82,31 % | 0,9517 |
| **RF v4 (final)** | `min_samples_leaf=2` | 97,34 % | **89,29 %** | **84,86 %** | **0,9595** |
| XGB v1 | `lr=0.3, max_depth=3` | 86,73 % | 86,37 % | 80,64 % | 0,9403 |
| XGB v2 | `lr=0.1, max_depth=6` | 89,64 % | 88,26 % | **83,68 %** | **0,9543** |


1. **`max_depth` es contraproducente en un Random Forest.** Las dos primeras
   versiones lo limitaban y ambas subajustaban.
2. **`min_samples_leaf` es el regulador adecuado**, y con 119.000 filas el valor
   bajo (2) funciona mejor que el conservador (10).
3. **Una accuracy de train alta no es un problema por sí sola.** La v4 tiene
   97 % en train y 89 % en test, y es la mejor de las cuatro. Lo que se optimiza
   es el test.
4. **En boosting, learning rate y número de árboles van juntos.** Bajar el
   primero sin subir el segundo deja el modelo infraentrenado, como demuestra la
   v3.
5. **Conviene mirar varias métricas.** La v1 tiene una precisión del 99,54 % que
   parece excelente hasta que se ve el recall del 36,74 %.

---
# Modelos finales

In [ ]:
modelo_rfc = Pipeline([
    ("preproceso", clone(preproceso)),
    ("clasificador", RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                            n_jobs=-1, random_state=42)),
])

modelo_xgb = Pipeline([
    ("preproceso", clone(preproceso)),
    ("clasificador", XGBClassifier(n_estimators=300,learning_rate=0.1,
                                   max_depth=6,n_jobs=-1,random_state=42)),
])

## Entrenamiento y evaluación del Random Forest

In [ ]:
modelo_rfc.fit(X_train,y_train)
print(modelo_rfc.score(X_test,y_test))

y_pred = modelo_rfc.predict(X_test)

# Obtención de las métricas de evaluación
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {acc:.2%}")
print(f"Precisión: {prec:.2%}")
print(f"Recall:    {rec:.2%}")
print(f"F1-Score:  {f1:.2%}")
print(f"ROC-AUC: {roc_auc_score(y_test, modelo_rfc.predict_proba(X_test)[:,1]):.4f}")

In [ ]:
# Comparamos valores reales con los valores del dummy
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Dummy accuracy:", round(dummy.score(X_test, y_test), 4))

El dummy es el suelo de referencia: sin él, un 89 % de accuracy no significa
nada.

In [ ]:
# Matriz de confusion y Curva ROC
ConfusionMatrixDisplay.from_estimator(
    modelo_rfc, X_test, y_test,
    display_labels=["No cancela", "Cancela"],
    cmap="Blues",
    normalize=None)
plt.title("Matriz de confusión - Random Forest")
plt.show()

In [ ]:
RocCurveDisplay.from_estimator(modelo_rfc, X_test, y_test, name="Random Forest")
plt.plot([0,1], [0,1], "k--", label="Azar (AUC=0.5)")
plt.title("Curva ROC")
plt.legend()
plt.show()

## Entrenamiento y evaluación del XGBoost

In [ ]:
modelo_xgb.fit(X_train,y_train)
print(modelo_xgb.score(X_test,y_test))

y_pred = modelo_xgb.predict(X_test)

# Obtención de las métricas de evaluación
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {acc:.2%}")
print(f"Precisión: {prec:.2%}")
print(f"Recall:    {rec:.2%}")
print(f"F1-Score:  {f1:.2%}")
print(f"ROC-AUC: {roc_auc_score(y_test, modelo_xgb.predict_proba(X_test)[:,1]):.4f}")

In [ ]:
# Matriz de confusion y Curva ROC
ConfusionMatrixDisplay.from_estimator(
    modelo_xgb, X_test, y_test,
    display_labels=["No cancela", "Cancela"],
    cmap="Blues",
    normalize=None)
plt.title("Matriz de confusión - Gradiant Boosting")
plt.show()

In [ ]:
RocCurveDisplay.from_estimator(modelo_xgb, X_test, y_test, name="Gradiant Boosting")
plt.plot([0,1], [0,1], "k--", label="Azar (AUC=0.5)")
plt.title("Curva ROC")
plt.legend()
plt.show()

## Curva ROC comparativa

In [ ]:
# Curva ROC comparativa de los dos modelos en un mismo gráfico
ax = plt.gca()
RocCurveDisplay.from_estimator(modelo_rfc, X_test, y_test, ax=ax, name="Random Forest")
RocCurveDisplay.from_estimator(modelo_xgb, X_test, y_test, ax=ax, name="Gradiant Boosting")
plt.plot([0,1], [0,1], "k--", label="Azar (AUC=0.5)")
plt.title("Curva ROC comparativa")
plt.legend()
plt.show()